In [1]:
import pandas as pd
from pathlib import Path
import os

TABLEAU_DIR = Path("tableau_export")
CURATED_DIR = Path("curated_data")
CLEANED_DIR = Path("cleaned_data")

print("Current folder:", Path.cwd())
print("Tableau files:", os.listdir(TABLEAU_DIR))

Current folder: C:\Users\user\Cloud Project
Tableau files: ['ai_services.csv', 'ai_taxonomy.csv', 'aws_gpu_summary.csv', 'gpu_pricing.csv', 'market_share.csv', 'region_master.csv', 'region_summary.csv', 'service_catalog.csv', 'service_category_summary.csv', 'service_summary.csv']


In [2]:
region = pd.read_csv(TABLEAU_DIR / "region_master.csv")

print(region.shape)
region.head()

(134, 6)


,provider,region_code,region_name,city,country,continent
0,AWS,af-south-1,Africa (Cape Town),Cape Town,South Africa,Africa
1,AWS,ap-east-1,Asia Pacific (Hong Kong),Hong Kong,Hong Kong,Asia Pacific
2,AWS,ap-east-2,Asia Pacific (Taipei),Taipei,Taiwan,Asia Pacific
3,AWS,ap-northeast-1,Asia Pacific (Tokyo),Tokyo,Japan,Asia Pacific
4,AWS,ap-northeast-2,Asia Pacific (Seoul),Seoul,South Korea,Asia Pacific


In [3]:
region.groupby(["provider", "continent"]).size()

provider  continent    
AWS       Africa            1
          Asia Pacific     14
          Canada            1
          Canada West       1
          Europe            8
          Israel            1
          Mexico            1
          Middle East       2
          South America     1
          US East           2
          US West           2
Azure     Unknown          57
GCP       APAC             12
          Africa            1
          Brazil            1
          Europe           13
          Middle East       3
          North America    12
          South America     1
dtype: int64

In [4]:
region = pd.read_csv("tableau_export/region_master.csv")

continent_fix = {

    # AWS
    "US East":"North America",
    "US West":"North America",
    "Canada":"North America",
    "Canada West":"North America",
    "Mexico":"North America",

    "Europe":"Europe",

    "Asia Pacific":"Asia",
    "Israel":"Middle East",
    "Middle East":"Middle East",

    "South America":"South America",

    "Africa":"Africa",

    # GCP
    "APAC":"Asia",
    "Brazil":"South America",
    "North America":"North America",

    # Azure
    "Unknown":"Unknown"
}

region["continent_clean"] = (
    region["continent"]
    .replace(continent_fix)
)

In [5]:
sorted(
    region["continent_clean"]
    .dropna()
    .unique()
)

['Africa',
 'Asia',
 'Europe',
 'Middle East',
 'North America',
 'South America',
 'Unknown']

In [8]:
import pandas as pd
from pathlib import Path
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time

TABLEAU_DIR = Path("tableau_export")

region = pd.read_csv(TABLEAU_DIR / "region_master.csv")

region.head()

,provider,region_code,region_name,city,country,continent
0,AWS,af-south-1,Africa (Cape Town),Cape Town,South Africa,Africa
1,AWS,ap-east-1,Asia Pacific (Hong Kong),Hong Kong,Hong Kong,Asia Pacific
2,AWS,ap-east-2,Asia Pacific (Taipei),Taipei,Taiwan,Asia Pacific
3,AWS,ap-northeast-1,Asia Pacific (Tokyo),Tokyo,Japan,Asia Pacific
4,AWS,ap-northeast-2,Asia Pacific (Seoul),Seoul,South Korea,Asia Pacific


In [9]:
continent_fix = {
    "US East":"North America",
    "US West":"North America",
    "Canada":"North America",
    "Canada West":"North America",
    "Mexico":"North America",
    "Europe":"Europe",
    "Asia Pacific":"Asia",
    "Israel":"Middle East",
    "Middle East":"Middle East",
    "South America":"South America",
    "Africa":"Africa",
    "APAC":"Asia",
    "Brazil":"South America",
    "North America":"North America",
    "Unknown":"Unknown"
}

region["continent_clean"] = region["continent"].replace(continent_fix)

region.groupby(["provider", "continent_clean"]).size()

provider  continent_clean
AWS       Africa              1
          Asia               14
          Europe              8
          Middle East         3
          North America       7
          South America       1
Azure     Unknown            57
GCP       Africa              1
          Asia               12
          Europe             13
          Middle East         3
          North America      12
          South America       2
dtype: int64

In [10]:
def make_geo_query(row):
    city = str(row["city"]) if pd.notna(row["city"]) else ""
    country = str(row["country"]) if pd.notna(row["country"]) else ""
    region_name = str(row["region_name"]) if pd.notna(row["region_name"]) else ""
    
    if city and city.lower() != "nan":
        return f"{city}, {country}"
    elif region_name and region_name.lower() != "nan":
        return f"{region_name}, {country}"
    else:
        return country

region["geo_query"] = region.apply(make_geo_query, axis=1)

region[["provider", "region_code", "region_name", "city", "country", "geo_query"]].head(20)

,provider,region_code,region_name,city,country,geo_query
0,AWS,af-south-1,Africa (Cape Town),Cape Town,South Africa,"Cape Town, South Africa"
1,AWS,ap-east-1,Asia Pacific (Hong Kong),Hong Kong,Hong Kong,"Hong Kong, Hong Kong"
2,AWS,ap-east-2,Asia Pacific (Taipei),Taipei,Taiwan,"Taipei, Taiwan"
3,AWS,ap-northeast-1,Asia Pacific (Tokyo),Tokyo,Japan,"Tokyo, Japan"
4,AWS,ap-northeast-2,Asia Pacific (Seoul),Seoul,South Korea,"Seoul, South Korea"
5,AWS,ap-northeast-3,Asia Pacific (Osaka),Osaka,Japan,"Osaka, Japan"
6,AWS,ap-south-1,Asia Pacific (Mumbai),Mumbai,India,"Mumbai, India"
7,AWS,ap-south-2,Asia Pacific (Hyderabad),Hyderabad,India,"Hyderabad, India"
8,AWS,ap-southeast-1,Asia Pacific (Singapore),Singapore,Singapore,"Singapore, Singapore"
9,AWS,ap-southeast-2,Asia Pacific (Sydney),Sydney,Australia,"Sydney, Australia"


In [11]:
geolocator = Nominatim(user_agent="cloud_region_geocoder_project")
geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1,
    max_retries=2,
    error_wait_seconds=2
)

unique_queries = region["geo_query"].dropna().unique()

geo_results = []

for q in unique_queries:
    try:
        location = geocode(q)
        if location:
            geo_results.append({
                "geo_query": q,
                "latitude": location.latitude,
                "longitude": location.longitude,
                "geocoded_address": location.address
            })
        else:
            geo_results.append({
                "geo_query": q,
                "latitude": None,
                "longitude": None,
                "geocoded_address": None
            })
    except Exception as e:
        geo_results.append({
            "geo_query": q,
            "latitude": None,
            "longitude": None,
            "geocoded_address": str(e)
        })

geo_df = pd.DataFrame(geo_results)

geo_df.head()

,geo_query,latitude,longitude,geocoded_address
0,"Cape Town, South Africa",-33.928830,18.417220,"Cape Town, City of Cape Town, Western Cape, 80..."
1,"Hong Kong, Hong Kong",22.281833,114.158283,"香港 Hong Kong, 中国"
2,"Taipei, Taiwan",25.037520,121.563680,"臺北市, 臺灣"
3,"Tokyo, Japan",35.676860,139.763895,"東京都, 日本"
4,"Seoul, South Korea",37.566679,126.978291,"서울특별시, 대한민국"


In [12]:
region_geo = region.merge(
    geo_df,
    on="geo_query",
    how="left"
)

region_geo[[
    "provider",
    "region_code",
    "region_name",
    "city",
    "country",
    "continent_clean",
    "latitude",
    "longitude"
]].head()

,provider,region_code,region_name,city,country,continent_clean,latitude,longitude
0,AWS,af-south-1,Africa (Cape Town),Cape Town,South Africa,Africa,-33.928830,18.417220
1,AWS,ap-east-1,Asia Pacific (Hong Kong),Hong Kong,Hong Kong,Asia,22.281833,114.158283
2,AWS,ap-east-2,Asia Pacific (Taipei),Taipei,Taiwan,Asia,25.037520,121.563680
3,AWS,ap-northeast-1,Asia Pacific (Tokyo),Tokyo,Japan,Asia,35.676860,139.763895
4,AWS,ap-northeast-2,Asia Pacific (Seoul),Seoul,South Korea,Asia,37.566679,126.978291


In [13]:
missing_geo = region_geo[
    region_geo["latitude"].isna() | region_geo["longitude"].isna()
][[
    "provider",
    "region_code",
    "region_name",
    "city",
    "country",
    "geo_query"
]]

missing_geo

,provider,region_code,region_name,city,country,geo_query


In [14]:
print("Missing Latitude:",
      region_geo["latitude"].isna().sum())

print("Missing Longitude:",
      region_geo["longitude"].isna().sum())

Missing Latitude: 0
Missing Longitude: 0


In [15]:
region_geo[[
    "provider",
    "region_code",
    "city",
    "country",
    "latitude",
    "longitude"
]].sample(20)

,provider,region_code,city,country,latitude,longitude
20,AWS,eu-south-1,Milan,Italy,45.464194,9.189635
82,Azure,uaenorth,Dubai,UAE,25.074282,55.188539
116,GCP,europe-west12,Turin,Italy,45.067755,7.682489
96,GCP,asia-northeast3,Seoul,South Korea,37.566679,126.978291
59,Azure,japaneast,"Tokyo, Saitama",Japan,35.788891,139.603626
28,AWS,mx-central-1,Central,Canada,53.540996,-113.492301
64,Azure,mexicocentral,Querétaro State,Mexico,20.273063,-99.881601
38,Azure,austriaeast,Vienna,Austria,48.208354,16.372504
88,Azure,westus,California,United States,36.701463,-118.755997
47,Azure,denmarkeast,Copenhagen,Denmark,55.686724,12.570072


In [16]:
region_geo.to_csv(
    "curated_data/region_geocoded_v1.csv",
    index=False
)

region_geo.to_csv(
    "tableau_export/region_geocoded.csv",
    index=False
)

In [17]:
region_geo["latitude"].isna().sum()
region_geo["longitude"].isna().sum()

np.int64(0)

In [18]:
region_geo.to_csv(
    "curated_data/region_geocoded_v1.csv",
    index=False
)

region_geo.to_csv(
    "tableau_export/region_geocoded.csv",
    index=False
)

In [19]:
aws_gpu = pd.read_csv(
    "cleaned_data/aws_gpu_availability_clean.csv"
)

print(aws_gpu.shape)

aws_gpu.head(20)

(85, 5)


,Instance Type,Instance Family,GPU,GPU Memory,accelerator
0,g7e.24xlarge,GPU instance,4.000,384 GB,Blackwell
1,g6.48xlarge,GPU instance,8.000,192 GB,L4
2,inf1.xlarge,Machine Learning ASIC Instances,1.000,NaN,Inferentia
3,inf1.2xlarge,Machine Learning ASIC Instances,1.000,NaN,Inferentia
4,gr6.8xlarge,GPU instance,1.000,24 GB,L4
5,g4dn.16xlarge,GPU instance,1.000,16 GB,T4
6,g6.8xlarge,GPU instance,1.000,24 GB,L4
7,g5.12xlarge,GPU instance,4.000,96 GB,A10G
8,g6e.12xlarge,GPU instance,4.000,192 GB,L4
9,g5.48xlarge,GPU instance,8.000,192 GB,A10G


In [20]:
aws_gpu["Instance Type"].tolist()[:50]

['g7e.24xlarge',
 'g6.48xlarge',
 'inf1.xlarge',
 'inf1.2xlarge',
 'gr6.8xlarge',
 'g4dn.16xlarge',
 'g6.8xlarge',
 'g5.12xlarge',
 'g6e.12xlarge',
 'g5.48xlarge',
 'g7e.12xlarge',
 'g4ad.8xlarge',
 'inf1.24xlarge',
 'g6.xlarge',
 'g6f.large',
 'g6f.2xlarge',
 'g6.4xlarge',
 'g6.24xlarge',
 'g7e.8xlarge',
 'p2.8xlarge',
 'g5.24xlarge',
 'g5.4xlarge',
 'g4dn.xlarge',
 'g4dn.8xlarge',
 'g6f.xlarge',
 'g6e.48xlarge',
 'g6f.4xlarge',
 'g6.2xlarge',
 'g7e.48xlarge',
 'gr6.4xlarge',
 'g6.16xlarge',
 'g4dn.2xlarge',
 'g4dn.4xlarge',
 'p5.48xlarge',
 'g5g.xlarge',
 'g7e.2xlarge',
 'inf2.8xlarge',
 'p4d.24xlarge',
 'p4de.24xlarge',
 'p3.2xlarge',
 'g3.8xlarge',
 'g6.12xlarge',
 'g7e.4xlarge',
 'p5.4xlarge',
 'g4ad.xlarge',
 'g6e.xlarge',
 'g6e.8xlarge',
 'g3.16xlarge',
 'inf1.6xlarge',
 'p6-b200.48xlarge']

In [21]:
benchmark_instances = [
    "p3.2xlarge",
    "p4d.24xlarge",
    "p5.48xlarge",
    "p6-b200.48xlarge",
    "g5.12xlarge",
    "g6.12xlarge",
    "inf2.8xlarge"
]

aws_gpu[
    aws_gpu["Instance Type"].isin(
        benchmark_instances
    )
]

,Instance Type,Instance Family,GPU,GPU Memory,accelerator
7,g5.12xlarge,GPU instance,4.0,96 GB,A10G
33,p5.48xlarge,GPU instance,8.0,640 GB HBM3,H100
36,inf2.8xlarge,Machine Learning ASIC Instances,1.0,32 GB,Inferentia 2
37,p4d.24xlarge,GPU instance,8.0,320 GB HBM2,A100
39,p3.2xlarge,GPU instance,1.0,16 GiB,V100
41,g6.12xlarge,GPU instance,4.0,96 GB,L4
49,p6-b200.48xlarge,GPU instance,8.0,1432 GB HBM3e,Blackwell (B200)


In [22]:
aws_gpu[
    aws_gpu["Instance Type"].isin(
        benchmark_instances
    )
][[
    "Instance Type",
    "GPU",
    "GPU Memory"
]]

,Instance Type,GPU,GPU Memory
7,g5.12xlarge,4.0,96 GB
33,p5.48xlarge,8.0,640 GB HBM3
36,inf2.8xlarge,1.0,32 GB
37,p4d.24xlarge,8.0,320 GB HBM2
39,p3.2xlarge,1.0,16 GiB
41,g6.12xlarge,4.0,96 GB
49,p6-b200.48xlarge,8.0,1432 GB HBM3e


In [23]:
import os

print("CURATED")
print(os.listdir("curated_data"))

print("\nTABLEAU")
print(os.listdir("tableau_export"))

CURATED
['ai_services_v2.csv', 'ai_taxonomy_summary_v2.csv', 'native_cloud_service_catalog_v2.csv', 'native_cloud_service_catalog_v3.csv', 'region_geocoded_v1.csv', 'service_category_summary_v3.csv', 'service_summary_v3.csv']

TABLEAU
['.ipynb_checkpoints', 'ai_services.csv', 'ai_taxonomy.csv', 'aws_gpu_summary.csv', 'gpu_pricing.csv', 'market_share.csv', 'region_geocoded.csv', 'region_master.csv', 'region_summary.csv', 'service_catalog.csv', 'service_category_summary.csv', 'service_summary.csv']


In [24]:
from pathlib import Path
import shutil

FINAL_DIR = Path("final_dataset")
FINAL_DIR.mkdir(exist_ok=True)

In [25]:
files_to_copy = [
    "market_share.csv",
    "region_geocoded.csv",
    "region_summary.csv",
    "gpu_pricing.csv",
    "aws_gpu_summary.csv",
    "ai_services.csv",
    "ai_taxonomy.csv",
    "service_catalog.csv",
    "service_summary.csv",
    "service_category_summary.csv"
]

for f in files_to_copy:
    shutil.copy(
        Path("tableau_export") / f,
        FINAL_DIR / f
    )

print("Done")

Done


In [26]:
import os

print(os.listdir("final_dataset"))

['ai_services.csv', 'ai_taxonomy.csv', 'aws_gpu_summary.csv', 'gpu_pricing.csv', 'market_share.csv', 'region_geocoded.csv', 'region_summary.csv', 'service_catalog.csv', 'service_category_summary.csv', 'service_summary.csv']


In [27]:
import pandas as pd
from pathlib import Path

FINAL_DIR = Path("final_dataset")

region = pd.read_csv(FINAL_DIR / "region_geocoded.csv")

country_to_continent = {
    "United States": "North America",
    "Canada": "North America",
    "Mexico": "North America",

    "Brazil": "South America",
    "Chile": "South America",

    "United Kingdom": "Europe",
    "France": "Europe",
    "Germany": "Europe",
    "Austria": "Europe",
    "Belgium": "Europe",
    "Denmark": "Europe",
    "Italy": "Europe",
    "Norway": "Europe",
    "Poland": "Europe",
    "Spain": "Europe",
    "Sweden": "Europe",
    "Switzerland": "Europe",
    "Europe": "Europe",

    "Australia": "Oceania",
    "New Zealand": "Oceania",

    "Japan": "Asia",
    "Korea": "Asia",
    "India": "Asia",
    "Indonesia": "Asia",
    "Malaysia": "Asia",
    "Asia Pacific": "Asia",

    "Israel": "Middle East",
    "Qatar": "Middle East",
    "UAE": "Middle East",
    "United Arab Emirates": "Middle East",
    "Saudi Arabia": "Middle East",

    "South Africa": "Africa"
}

region["continent_clean"] = region["country"].map(country_to_continent).fillna(region["continent_clean"])

region_summary_fixed = (
    region
    .groupby("provider")
    .agg(
        region_count=("region_code", "nunique"),
        country_count=("country", "nunique"),
        continent_count=("continent_clean", "nunique")
    )
    .reset_index()
)

region.to_csv(FINAL_DIR / "region_geocoded.csv", index=False)
region_summary_fixed.to_csv(FINAL_DIR / "region_summary.csv", index=False)

print(region.groupby(["provider", "continent_clean"]).size())
print(region_summary_fixed)

provider  continent_clean
AWS       Africa              1
          Asia               11
          Europe              8
          Middle East         3
          North America       7
          Oceania             3
          South America       1
Azure     Africa              2
          Asia               12
          Europe             19
          Middle East         4
          North America      12
          Oceania             5
          South America       3
GCP       Africa              1
          Asia               10
          Europe             13
          Middle East         3
          North America      12
          Oceania             2
          South America       2
dtype: int64
  provider  region_count  country_count  continent_count
0      AWS            34             26                7
1    Azure            57             30                7
2      GCP            43             38                7


In [28]:
import os
from pathlib import Path

for f in [
    "region_geocoded.csv",
    "region_summary.csv"
]:
    p = Path("final_dataset") / f
    print(f)
    print("Modified:", p.stat().st_mtime)

region_geocoded.csv
Modified: 1780590622.8436828
region_summary.csv
Modified: 1780590622.8456833


In [30]:
import pandas as pd

market_share = pd.read_csv(
    TABLEAU_DIR / "market_share.csv"
)

market_share.head()

,year_quarter,provider,market_share
0,2024 Q4,AWS,30
1,2025 Q1,AWS,32
2,2025 Q2,AWS,30
3,2025 Q3,AWS,29
4,2025 Q4,AWS,28


In [32]:
market_latest = (
    market_share
    .sort_values("year_quarter")
    .groupby("provider")
    .tail(1)
)

market_latest

,year_quarter,provider,market_share
11,2026 Q1,Azure,21
5,2026 Q1,AWS,28
17,2026 Q1,GCP,15


In [33]:
region_summary = pd.read_csv(
    TABLEAU_DIR / "region_summary.csv"
)

ai_services = pd.read_csv(
    TABLEAU_DIR / "ai_services.csv"
)

In [34]:
print(region_summary.columns)
print(ai_services.columns)

Index(['provider', 'region_count', 'country_count', 'continent_count'], dtype='object')
Index(['provider', 'service_code', 'service_name', 'service_title', 'category',
       'source', 'collection_date', 'category_v2', 'category_v3',
       'service_origin', 'ai_type', 'ai_taxonomy'],
      dtype='object')


In [35]:
import pandas as pd

market_share = pd.read_csv(TABLEAU_DIR / "market_share.csv")
region_summary = pd.read_csv(TABLEAU_DIR / "region_summary.csv")
ai_services = pd.read_csv(TABLEAU_DIR / "ai_services.csv")

market_latest = (
    market_share
    .sort_values("year_quarter")
    .groupby("provider")
    .tail(1)
    [["provider", "year_quarter", "market_share"]]
)

ai_count = (
    ai_services
    .groupby("provider")["service_name"]
    .nunique()
    .reset_index(name="ai_service_count")
)

position_matrix = (
    region_summary[["provider", "region_count", "country_count"]]
    .merge(ai_count, on="provider", how="left")
    .merge(market_latest, on="provider", how="left")
)

position_matrix.to_csv(
    TABLEAU_DIR / "ai_market_position.csv",
    index=False
)

position_matrix

,provider,region_count,country_count,ai_service_count,year_quarter,market_share
0,AWS,34,26,6,2026 Q1,28
1,Azure,57,30,4,2026 Q1,21
2,GCP,43,38,6,2026 Q1,15


In [36]:
position_matrix.to_csv(
    TABLEAU_DIR / "ai_market_position.csv",
    index=False
)

In [37]:
import os
os.listdir(TABLEAU_DIR)

['.ipynb_checkpoints',
 'ai_market_position.csv',
 'ai_services.csv',
 'ai_taxonomy.csv',
 'aws_gpu_summary.csv',
 'gpu_pricing.csv',
 'market_share.csv',
 'region_geocoded.csv',
 'region_master.csv',
 'region_summary.csv',
 'service_catalog.csv',
 'service_category_summary.csv',
 'service_summary.csv']

In [42]:
import pandas as pd

service_category_summary = pd.read_csv(
    TABLEAU_DIR / "service_category_summary.csv"
)

service_category_summary.head()

,provider,category,service_count
0,AWS,AI / Machine Learning,14
1,AWS,Analytics,8
2,AWS,Billing / Support,5
3,AWS,Compute,8
4,AWS,Database,3


In [43]:
service_category_summary.columns

Index(['provider', 'category', 'service_count'], dtype='object')

In [44]:
service_category_summary.head(20)

,provider,category,service_count
0,AWS,AI / Machine Learning,14
1,AWS,Analytics,8
2,AWS,Billing / Support,5
3,AWS,Compute,8
4,AWS,Database,3
5,AWS,DevOps,5
6,AWS,Management,3
7,AWS,Messaging / Integration,5
8,AWS,Networking,5
9,AWS,Security,7
